# Lesson 04 - Σχέδιο Χρήσης Εργαλείων

Σε αυτό το μάθημα θα μάθετε το σχέδιο σχεδιασμού **Χρήση Εργαλείων** για πράκτορες AI χρησιμοποιώντας το Microsoft Agent Framework (Python). Καλύπτουμε:

- Ορισμό λειτουργιών εργαλείων με το διακοσμητή `@tool` και τυποποιημένες παραμέτρους
- Παροχή σχημάτων εργαλείων ώστε το μοντέλο να γνωρίζει τι κάνει κάθε εργαλείο
- Έλεγχο εκτέλεσης εργαλείων με το `approval_mode`
- Επιστροφή **δομημένης εξόδου** μέσω μοντέλων Pydantic και `response_format`

Το σενάριο είναι ένας **πράκτορας κρατήσεων ταξιδίων** που μπορεί να αναζητήσει προορισμούς, να ελέγξει διαθεσιμότητα και να ανακτήσει πληροφορίες πτήσεων.


## Ρύθμιση


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Ορισμός Εργαλείων με τον Διακοσμητή @tool

Ο διακοσμητής `@tool` μετατρέπει μια απλή συνάρτηση Python σε ένα εργαλείο που μπορεί να καλεί ένας πράκτορας.
Κύρια σημεία:

- Το **docstring** γίνεται η περιγραφή του εργαλείου που βλέπει το μοντέλο.
- Οι **δηλώσεις τύπου** (συμπεριλαμβανομένου του `Annotated` με περιγραφές) ορίζουν το σχήμα του εργαλείου.
- Το `approval_mode` ελέγχει αν ο χρήστης πρέπει να εγκρίνει κάθε κλήση πριν εκτελεστεί.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## Δημιουργία ενός Πράκτορα με Πολλαπλά Εργαλεία

Περάστε και τα τρία εργαλεία στον πελάτη ώστε το μοντέλο να μπορεί να καλεί όποιο από αυτά χρειάζεται για να απαντήσει στην ερώτηση του χρήστη.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = await provider.create_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## Δομημένη Έξοδος με Εργαλεία

Ορίζοντας το `response_format` σε ένα μοντέλο Pydantic, ο πράκτορας αναγκάζεται να επιστρέψει ένα καλά τυποποιημένο αντικείμενο JSON αντί για ελεύθερο κείμενο. Αυτό είναι χρήσιμο όταν ο κώδικας που ακολουθεί χρειάζεται να καταναλώσει το αποτέλεσμα προγραμματιστικά.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = await provider.create_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## Πρότυπα Έγκρισης Εργαλείων

Η παράμετρος `approval_mode` στο `@tool` ελέγχει εάν οι κλήσεις εργαλείων απαιτούν ανθρώπινη έγκριση προτού εκτελεστούν:

| Mode | Συμπεριφορά |
|---|---|
| `"never_require"` | Το εργαλείο τρέχει αυτόματα — δεν απαιτείται επιβεβαίωση από τον χρήστη. |
| `"always_require"` | Κάθε κλήση πρέπει να εγκριθεί από τον χρήστη πριν εκτελεστεί. |

Χρησιμοποιήστε το `"always_require"` για εργαλεία που έχουν παρενέργειες (π.χ. κράτηση πτήσης, χρέωση πιστωτικής κάρτας) ώστε να υπάρχει ανθρώπινη επίβλεψη.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## Περίληψη

Σε αυτό το μάθημα μάθατε πώς να:

1. **Ορίζετε εργαλεία** χρησιμοποιώντας τον διακοσμητή `@tool` με τυποποιημένες παραμέτρους και docstrings που λειτουργούν ως το σχήμα του εργαλείου.
2. **Συνθέτετε πολλαπλά εργαλεία** έτσι ώστε ο πράκτορας να μπορεί να τα καλεί κατά σειρά για να απαντά σε πολύπλοκα ερωτήματα.
3. **Επιστρέφετε δομημένη έξοδο** περνώντας ένα μοντέλο Pydantic ως `response_format`.
4. **Ελέγχετε την έγκριση του εργαλείου** με το `approval_mode` για να διατηρείται ένας άνθρωπος στη διαδικασία για ευαίσθητες λειτουργίες.

Αυτά τα πρότυπα αποτελούν τη βάση για την κατασκευή αξιόπιστων, παραγωγικά έτοιμων πρακτόρων που μπορούν να αλληλεπιδρούν με εξωτερικά συστήματα με ασφάλεια.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Αποποίηση ευθυνών**:  
Αυτό το έγγραφο έχει μεταφραστεί χρησιμοποιώντας την υπηρεσία μετάφρασης με τεχνητή νοημοσύνη [Co-op Translator](https://github.com/Azure/co-op-translator). Παρόλο που προσπαθούμε για ακρίβεια, παρακαλούμε να λάβετε υπόψη ότι οι αυτοματοποιημένες μεταφράσεις ενδέχεται να περιέχουν λάθη ή ανακρίβειες. Το αρχικό έγγραφο στη γλώσσα του θεωρείται η επίσημη πηγή. Για κρίσιμες πληροφορίες συνιστάται επαγγελματική ανθρώπινη μετάφραση. Δεν φέρουμε καμία ευθύνη για τυχόν παρεξηγήσεις ή λανθασμένες ερμηνείες που προκύπτουν από τη χρήση αυτής της μετάφρασης.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
